# JalVaani TFT v4 — groundwater forecasting (t+7 / t+14 / t+30), Colab run

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

**Upload in Step 1** (all from `Models/TFT/` on your laptop):

| file | what it is |
|---|---|
| `tft_model_v3.py` | the TFT network (unchanged) |
| `tft_v4.py` | the v4 pipeline: fixed evaluation, seasonal-naive baseline, calibration, ensemble |
| `master_dataset_v4target.csv` | **the dataset to use** — rebuilt groundwater target (25 districts, 2023-01-01 → 2026-06-04) |
| `master_dataset_v3target.csv` | *optional* — the old target, only for the before/after comparison |

**What it does**
1. **Dev screening** on folds 1-4: a few candidate set-ups, every one scored against persistence *and* the seasonal-naive baseline.
2. **Picks the best** by a rule fixed in advance (lowest mean MAE ratio vs seasonal-naive).
3. **Final run** of that set-up on all 6 folds with 5 seeds; folds 5-6 (Oct 2025 → Jun 2026) are reported separately as the untouched test.
4. Downloads `tft_v4_outputs.zip`.

Time on a T4: about 60-75 min for everything, or about 20 min with `QUICK = True`, which skips screening and runs the recommended set-up straight away.

In [ ]:
# ── STEP 1: upload the files listed above ────────────────────────────────────
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install -q lightgbm

In [ ]:
# ── STEP 2: setup ────────────────────────────────────────────────────────────
import sys, copy, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
sys.path.insert(0, '/content')
import tft_v4 as v4

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then re-run from the top'
print('GPU:', torch.cuda.get_device_name(0), '| pipeline', v4.PIPELINE_VERSION)

DATA_NEW = '/content/master_dataset_v4target.csv'
DATA_OLD = '/content/master_dataset_v3target.csv'
OUT      = '/content/outputs/v4'
assert Path(DATA_NEW).exists(), 'master_dataset_v4target.csv was not uploaded'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

In [ ]:
# ── STEP 3: settings ─────────────────────────────────────────────────────────
QUICK       = False                      # True = skip dev screening, run the recommended set-up only
DEV_SEEDS   = [42, 1337, 2026]
FINAL_SEEDS = [42, 1337, 2026, 7, 99]
RECOMMENDED = 'snaive'                   # used when QUICK = True

# Candidate set-ups (only the TRAINING settings cost GPU time — calibration / ensemble
# variants of the same networks reuse the cached predictions and take seconds).
CANDIDATES = {
    # v3 model + the v4 evaluation fixes (reference)
    'v3fixed':     dict(),
    # same networks, v4 calibration (season-weighted pool + adaptive conformal) + stacked ensemble
    'v4post':      dict(calib_mode='v4', combine='stack'),
    # network predicts the residual on top of the seasonal-naive forecast
    'snaive':      dict(target_mode='snaive', calib_mode='v4', combine='stack'),
    # ... with extra loss weight on days 7 / 14 / 30
    'snaive_lw':   dict(target_mode='snaive', calib_mode='v4', combine='stack', loss_weighting='eval_steps'),
    # ... without the ERA5 + Sentinel inputs (they showed no signal beyond seasonality)
    'snaive_lean': dict(target_mode='snaive', calib_mode='v4', combine='stack', feature_groups=['gwl', 'rain', 'met']),
}

def make(tag, mode='dev', data=DATA_NEW, seeds=DEV_SEEDS, **kw):
    cfg = v4.Config(tag=tag, mode=mode, data_path=data, out_root=OUT, seeds=list(seeds))
    for k, v in kw.items():
        setattr(cfg, k, v)
    return cfg

def brief(tag, S):
    rows = []
    for name, sm in S.items():
        h, c = sm['head'], sm['cov']
        rows.append({'run': tag, 'model': name,
                     **{f'rMAE_P {k}': h.loc[k, 'rMAE_P'] for k in v4.HORDER},
                     **{f'rMAE_SN {k}': h.loc[k, 'rMAE_SN'] for k in v4.HORDER},
                     **{f'cov {k}': c.loc[k, 'Cov (%)'] for k in v4.HORDER},
                     'mean rMAE_SN': h['rMAE_SN'].mean()})
    return pd.DataFrame(rows)

## Dev screening (folds 1-4)
Each candidate is trained on folds 1-4 only. Folds 5-6 are not touched here.

In [ ]:
SCREEN = []
t0 = time.time()
if not QUICK:
    if Path(DATA_OLD).exists():
        print('######## reference on the OLD target ########')
        res = v4.run(make('dev_v3fixed_oldtarget', data=DATA_OLD), verbose=False)
        SCREEN.append(brief('v3fixed (old target)', v4.report(res, verbose=False)))
    for tag, kw in CANDIDATES.items():
        print(f'######## dev: {tag}   [{(time.time()-t0)/60:.0f} min elapsed] ########')
        res = v4.run(make('dev_' + tag, **kw), verbose=False)
        SCREEN.append(brief(tag, v4.report(res, verbose=False)))
        display(SCREEN[-1].round(3))
    SCREEN = pd.concat(SCREEN, ignore_index=True)
    SCREEN.to_csv(f'{OUT}/dev_screening_summary.csv', index=False)
    print(f'\nScreening done in {(time.time()-t0)/60:.1f} min')
    display(SCREEN.round(3))

In [ ]:
# ── pick the set-up for the final run (rule fixed in advance) ────────────────
# lowest mean (over t+7/14/30) MAE ratio vs seasonal-naive of the final forecast
# (the Ensemble where one exists, otherwise the TFT), on the NEW target, dev folds.
if QUICK:
    BEST = RECOMMENDED
else:
    cand = SCREEN[SCREEN['run'].isin(CANDIDATES)]
    final_rows = cand[cand['model'] == 'Ensemble']
    final_rows = pd.concat([final_rows, cand[(cand['model'] == 'TFT') & ~cand['run'].isin(final_rows['run'])]])
    BEST = final_rows.sort_values('mean rMAE_SN').iloc[0]['run']
    display(final_rows.sort_values('mean rMAE_SN')[['run', 'model', 'mean rMAE_SN'] +
                                                    [f'rMAE_P {k}' for k in v4.HORDER]].round(3))
print('Final set-up:', BEST, CANDIDATES[BEST])

## Final run — all 6 folds, 5 seeds
Folds 1-4 also feed the calibration pool of later folds, so all six are trained. Results are reported twice:
**all 6 folds** (comparable with the v3 write-up), and **folds 5-6 only** (the locked test set nothing was tuned on).

In [ ]:
FINAL_TAG = 'FINAL_' + BEST
res = v4.run(make(FINAL_TAG, mode='all', seeds=FINAL_SEEDS, **CANDIDATES[BEST]), verbose=True)
print('\n\n================  ALL 6 FOLDS  ================')
S_all = v4.report(res, verbose=True)

In [ ]:
def subset(res, folds, tag):
    r = dict(res)
    for k in ('daily', 'weekly', 'base', 'keys', 'all'):
        r[k] = res[k][res[k]['fold'].isin(folds)].reset_index(drop=True)
    cfg = copy.deepcopy(res['cfg']); cfg.tag = tag; cfg.mode = 'final'
    r['cfg'] = cfg
    r['wins'] = res['wins'][res['wins']['fold'].isin(folds)]
    r['out_dir'] = Path(OUT) / tag; r['out_dir'].mkdir(parents=True, exist_ok=True)
    return r

LOCKED = ['fold5_postmonsoon2025', 'fold6_dry2026']
print('================  LOCKED TEST: folds 5-6 (Oct 2025 -> Jun 2026)  ================')
S_locked = v4.report(subset(res, LOCKED, FINAL_TAG + '_locked'), verbose=True)

## Results to quote
`rMAE_P` = error ÷ error of "no change" (persistence). `rMAE_SN` = error ÷ error of the seasonal-naive forecast.
**Below 1 means better than that baseline.** Coverage should be about 80%.

In [ ]:
def quote(S, label):
    rows = []
    for name, sm in S.items():
        for h in v4.HORDER:
            hd, cv, gp = sm['head'].loc[h], sm['cov'].loc[h], sm['gap'].get(h, {})
            sig = sm['sig'].set_index(['horizon', 'vs'])
            rows.append({'test set': label, 'model': name, 'horizon': h,
                         'MAE (m)': hd['MAE (m)'], 'RMSE (m)': hd['RMSE (m)'], 'MAPE (%)': hd['MAPE (%)'],
                         'rMAE vs persistence': hd['rMAE_P'], 'rMAE vs seasonal': hd['rMAE_SN'],
                         'p (vs persistence)': sig.loc[(h, 'persistence'), 'p_boot_holm'],
                         'coverage 80% (%)': cv['Cov (%)'], 'width (m)': cv['Width (m)'],
                         'districts beating persistence': f"{gp.get('n_lt1', np.nan):.0f}/{gp.get('n', np.nan):.0f}",
                         'worst5 - best5 gap': gp.get('gap', np.nan)})
    return pd.DataFrame(rows)

QUOTE = pd.concat([quote(S_locked, 'locked folds 5-6'), quote(S_all, 'all 6 folds')], ignore_index=True)
QUOTE.to_csv(f'{OUT}/RESULTS_TO_QUOTE.csv', index=False)
display(QUOTE.round(3))

In [ ]:
# ── plots ────────────────────────────────────────────────────────────────────
D = res['daily']
model = 'Ensemble' if 'Ensemble' in S_all else 'TFT'
pdist = S_all[model]['per_dist']
TIER_COLOR = {'A': '#1b7837', 'B': '#7fbf7b', 'C': '#d6604d', 'D': '#999999'}
for h in v4.HORDER:
    s = pdist[pdist.Horizon == h].sort_values('rMAE_P')
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(s['district'], s['rMAE_P'], color=[TIER_COLOR.get(t, '#999') for t in s['tier']])
    ax.axvline(1, color='k', ls='--'); ax.invert_yaxis(); ax.grid(axis='x', alpha=.3)
    ax.set_xlabel('error / persistence error  (< 1 = better than "no change")')
    ax.set_title(f'{model} per-district skill @ {h}, all 6 folds (colour = data tier)')
    plt.tight_layout(); plt.savefig(f'{OUT}/{FINAL_TAG}/per_district_{h.replace("+", "")}.png', dpi=130); plt.show()

cal = model + ' (calibrated)'
for dist in ['Nagpur', 'Wardha', 'Washim', 'Akola']:
    s = D[(D.model == cal) & (D.district == dist) & (D.horizon == 't+30')].sort_values('target_date')
    if not len(s):
        continue
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(s.target_date, s.q10, s.q90, alpha=.2, color='steelblue', label='80% interval')
    ax.plot(s.target_date, s.q50, color='steelblue', lw=1.5, label=f'{model} forecast')
    ax.plot(s.target_date, s.persistence, color='darkorange', lw=1, ls=':', label='no-change')
    ax.plot(s.target_date, s.actual, color='k', lw=1.3, ls='--', label='actual')
    ax.set_title(f'{dist} — 30-day-ahead forecast, all test folds'); ax.set_ylabel('GWL (m)')
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f'{OUT}/{FINAL_TAG}/forecast_{dist.lower()}.png', dpi=130); plt.show()

In [ ]:
# ── download everything (send this zip back) ─────────────────────────────────
shutil.rmtree(f'{OUT}/_cache', ignore_errors=True)        # cached predictions, large and not needed
shutil.make_archive('/content/tft_v4_outputs', 'zip', OUT)
files.download('/content/tft_v4_outputs.zip')